# Limpieza y normalización · venta_idealista.csv

Objetivo: cargar el dataset y construir una cadena de transformaciones **sin modificar el DataFrame original**.

Estrategia:
- `df_raw`: lectura tal cual
- `df_work`, `df_bool_norm`, ...: pasos de limpieza incrementales (copias)


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)

## 1) Carga del CSV

Si el CSV tiene comillas rotas o líneas corruptas, usamos `on_bad_lines="warn"` para **no bloquear** el análisis.

In [2]:
df_raw = pd.read_csv(
    "../data/raw/scraping_manual/venta_idealista.csv",
    delimiter = ";",
    on_bad_lines="warn",
    encoding="utf-8",
    encoding_errors="replace",
    skipinitialspace=True,
    index_col=0   # ← primera columna como índice
)

df_raw.head()

,fuente,tipo_inmueble,subtipo,direccion_texto,barrio_zona,municipio,precio_eur,precio_anterior_eur,descuento_pct,garaje_incluido,habitaciones,superficie_m2,planta,exterior,ascensor,obra_nueva_flag
id_anuncio,,,,,,,,,,,,,,,,
1,Inmobiliaria E-Castro,piso,NaN,"""Avenida Riomar, 1",Cotolino,Castro-Urdiales,498000.0,NaN,NaN,sí,4.0,121.0,4,sí,sí,0.0
2,Inmobiliarias.com,casa,chalét independiente,"Barrio Brazomar, 46",Cotolino,Castro-Urdiales,575000.0,NaN,NaN,sí,5.0,232.0,NaN,desconocido,desconocido,0.0
3,Inmobiliaria E-Castro,piso,NaN,"""Calle Leonardo Rucabado, 6",Centro,Castro-Urdiales,299000.0,NaN,NaN,desconocido,2.0,84.0,3,sí,sí,0.0
4,Inmobiliaria E-Castro,piso,NaN,"""Calle de la Ronda, 32",Centro,Castro-Urdiales,340000.0,NaN,NaN,desconocido,4.0,99.0,5,sí,sí,0.0
5,Inmobiliaria E-Castro,casa,chalét independiente,Barrio Sámano,Sámano,Castro-Urdiales,799000.0,NaN,NaN,sí,5.0,511.0,NaN,desconocido,desconocido,0.0


In [3]:
# Primer DataFrame de trabajo (no modifica el original)
df_work = df_raw.copy()
df_work = df_work.rename(columns={"obra_nueva_flag": "obra_nueva"})
df_work.info()

<class 'pandas.DataFrame'>
RangeIndex: 861 entries, 1 to 861
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   fuente               861 non-null    str    
 1   tipo_inmueble        861 non-null    str    
 2   subtipo              415 non-null    str    
 3   direccion_texto      858 non-null    str    
 4   barrio_zona          694 non-null    str    
 5   municipio            846 non-null    str    
 6   precio_eur           860 non-null    float64
 7   precio_anterior_eur  66 non-null     float64
 8   descuento_pct        66 non-null     float64
 9   garaje_incluido      781 non-null    str    
 10  habitaciones         809 non-null    float64
 11  superficie_m2        762 non-null    float64
 12  planta               377 non-null    str    
 13  exterior             714 non-null    str    
 14  ascensor             728 non-null    str    
 15  obra_nueva           860 non-null    float64
dtypes

## 2) Normalización de columnas booleanas

Convertimos `sí/no/desconocido` a `1/0/NaN` y lo dejamos como numérico.


In [4]:
df_bool_norm = df_work.copy()

cols_bool = ["ascensor", "exterior", "garaje_incluido"]

for col in cols_bool:
    if col not in df_bool_norm.columns:
        continue

    df_bool_norm[col] = (
        df_bool_norm[col]
            .astype(str)
            .str.strip()
            .str.lower()
            .replace({
                "desconocido": np.nan,
                "nan": np.nan,
                "sí": 1,
                "si": 1,
                "no": 0
            })
    )
    df_bool_norm[col] = pd.to_numeric(df_bool_norm[col], errors="coerce")
df_bool_norm.head()


,fuente,tipo_inmueble,subtipo,direccion_texto,barrio_zona,municipio,precio_eur,precio_anterior_eur,descuento_pct,garaje_incluido,habitaciones,superficie_m2,planta,exterior,ascensor,obra_nueva
id_anuncio,,,,,,,,,,,,,,,,
1,Inmobiliaria E-Castro,piso,NaN,"""Avenida Riomar, 1",Cotolino,Castro-Urdiales,498000.0,NaN,NaN,1.0,4.0,121.0,4,1.0,1.0,0.0
2,Inmobiliarias.com,casa,chalét independiente,"Barrio Brazomar, 46",Cotolino,Castro-Urdiales,575000.0,NaN,NaN,1.0,5.0,232.0,NaN,NaN,NaN,0.0
3,Inmobiliaria E-Castro,piso,NaN,"""Calle Leonardo Rucabado, 6",Centro,Castro-Urdiales,299000.0,NaN,NaN,NaN,2.0,84.0,3,1.0,1.0,0.0
4,Inmobiliaria E-Castro,piso,NaN,"""Calle de la Ronda, 32",Centro,Castro-Urdiales,340000.0,NaN,NaN,NaN,4.0,99.0,5,1.0,1.0,0.0
5,Inmobiliaria E-Castro,casa,chalét independiente,Barrio Sámano,Sámano,Castro-Urdiales,799000.0,NaN,NaN,1.0,5.0,511.0,NaN,NaN,NaN,0.0


## 3) Limpieza de `planta`

Normalizamos 'bajo' a 0 y convertimos a numérico.


In [5]:
df_planta = df_bool_norm.copy()
print(f"Antes de la limpieza:\n{df_planta["planta"].unique()}")
if "planta" in df_planta.columns:
    df_planta["planta"] = (
        df_planta["planta"]
            .astype(str)
            .str.strip()
            .replace({"bajo": "0", "Bajo": "0"})
    )
    df_planta["planta"] = pd.to_numeric(df_planta["planta"], errors="coerce")

print(f"\nDespués de la limpieza:\n{df_planta["planta"].unique()}")

Antes de la limpieza:
<StringArray>
[          '4',           nan,           '3',           '5',           '6',
           '2',           '1',           '8',        'bajo',          '10',
           '9',          '13',           '7',          '15', 'entreplanta']
Length: 15, dtype: str

Después de la limpieza:
[ 4. nan  3.  5.  6.  2.  1.  8.  0. 10.  9. 13.  7. 15.]


## 4) Normalización de tipo y subtipo

Tranformamos el tipo de las fincas a casas y añadimos columna ´finca´.

In [6]:
print(f"Total inmuebles del tipo finca = {len(df_planta.loc[df_planta["tipo_inmueble"].isin(["finca"])])}")

print(df_planta[["tipo_inmueble", "subtipo", "superficie_m2", "precio_eur"]].loc[df_planta["tipo_inmueble"].isin(["finca"])].head(6))

df_finca = df_planta.copy()

df_finca["finca"] = np.where(df_finca["tipo_inmueble"].isin(["finca"]), 1, 0)
df_finca["tipo_inmueble"] = np.where(df_finca["finca"] == True, "casa", df_finca["tipo_inmueble"])

print(df_finca[["tipo_inmueble", "subtipo", "superficie_m2", "precio_eur"]].loc[df_finca["finca"]==1].head(6))

Total inmuebles del tipo finca = 1
           tipo_inmueble        subtipo  superficie_m2  precio_eur
id_anuncio                                                        
724                finca  finca rústica          574.0   1900000.0
           tipo_inmueble        subtipo  superficie_m2  precio_eur
id_anuncio                                                        
724                 casa  finca rústica          574.0   1900000.0


In [7]:
df_subtipo_normalizado = df_finca.copy()

for col in ["tipo_inmueble", "subtipo"]:
    if col in df_subtipo_normalizado.columns:
        df_subtipo_normalizado[col] = df_subtipo_normalizado[col].astype(str).str.lower().str.strip().replace({"nan": np.nan})

print(f"Registros únicos del tipo:\n{df_subtipo_normalizado["tipo_inmueble"].unique()}")
print(f"\nRegistros únicos del subtipo:\n{df_subtipo_normalizado["subtipo"].unique()}")

Registros únicos del tipo:
<StringArray>
[         'piso',          'casa',         'ático',        'dúplex',
      'edificio', 'finca_rustica',  'garaje/local',     'piso/casa',
        'garaje']
Length: 9, dtype: str

Registros únicos del subtipo:
<StringArray>
[                          nan,        'chalét independiente',
              'chalét adosado',              'chalét pareado',
                 'apartamento',              'casa de piedra',
                      'dúplex',              'casa de pueblo',
          'casa independiente',                        'casa',
                      'chalét',                     'estudio',
                       'masía',                       'villa',
              'casa montañesa',                      'casona',
                       'ático',                  'casa rural',
                'casa indiana',            'casa unifamiliar',
                'casa adosada',                     'palacio',
                       'torre',            

Estudiamos el caso particular de tipo = piso/casa

In [8]:
df_subtipo_normalizado.loc[df_subtipo_normalizado["tipo_inmueble"]=="piso/casa"].head()
# Descartamos por carecer de superficie y habitaciones

df_tipo_normalizado = df_subtipo_normalizado.loc[df_subtipo_normalizado["tipo_inmueble"]!="piso/casa"].copy()
df_tipo_normalizado.head()

,fuente,tipo_inmueble,subtipo,direccion_texto,barrio_zona,municipio,precio_eur,precio_anterior_eur,descuento_pct,garaje_incluido,habitaciones,superficie_m2,planta,exterior,ascensor,obra_nueva,finca
id_anuncio,,,,,,,,,,,,,,,,,
1,Inmobiliaria E-Castro,piso,NaN,"""Avenida Riomar, 1",Cotolino,Castro-Urdiales,498000.0,NaN,NaN,1.0,4.0,121.0,4.0,1.0,1.0,0.0,0
2,Inmobiliarias.com,casa,chalét independiente,"Barrio Brazomar, 46",Cotolino,Castro-Urdiales,575000.0,NaN,NaN,1.0,5.0,232.0,NaN,NaN,NaN,0.0,0
3,Inmobiliaria E-Castro,piso,NaN,"""Calle Leonardo Rucabado, 6",Centro,Castro-Urdiales,299000.0,NaN,NaN,NaN,2.0,84.0,3.0,1.0,1.0,0.0,0
4,Inmobiliaria E-Castro,piso,NaN,"""Calle de la Ronda, 32",Centro,Castro-Urdiales,340000.0,NaN,NaN,NaN,4.0,99.0,5.0,1.0,1.0,0.0,0
5,Inmobiliaria E-Castro,casa,chalét independiente,Barrio Sámano,Sámano,Castro-Urdiales,799000.0,NaN,NaN,1.0,5.0,511.0,NaN,NaN,NaN,0.0,0


Normalizamos planta = 0 para inmuebles tipo casa

In [9]:
df_planta_casa = df_tipo_normalizado.copy()

df_planta_casa["planta"] = np.where(df_planta_casa["tipo_inmueble"]=="casa", 0, df_planta_casa["planta"])

pd.crosstab(df_planta_casa["tipo_inmueble"], df_planta_casa["planta"])

planta,0.0,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,10.0,13.0,15.0
tipo_inmueble,,,,,,,,,,,,,
casa,406,0,0,0,0,0,0,0,0,0,0,0,0
dúplex,1,3,4,2,2,4,0,0,0,0,0,0,0
edificio,0,1,0,0,0,0,0,0,0,0,0,0,0
piso,89,76,70,46,25,20,7,3,3,2,1,1,1
ático,0,0,1,2,7,3,1,0,0,0,0,0,0


Eliminamos registros cuyo `tipo_inmueble` no es vivienda (según la lista).


In [10]:
(df_planta_casa[["tipo_inmueble", "subtipo", "superficie_m2", "precio_eur"]]
 .loc[df_planta_casa["tipo_inmueble"].isin(["finca_rustica", "garaje", "garaje/local", "edificio"])].head(15))

,tipo_inmueble,subtipo,superficie_m2,precio_eur
id_anuncio,,,,
47,edificio,NaN,860.0,780000.0
232,finca_rustica,NaN,622.0,229000.0
481,finca_rustica,NaN,106.0,179000.0
575,finca_rustica,barrio rocillo,220.0,320000.0
577,finca_rustica,n-629a,300.0,89000.0
584,finca_rustica,camino de solorga,200.0,300000.0
769,garaje/local,NaN,NaN,18000.0
848,garaje,NaN,NaN,25000.0


In [11]:
df_limpio = df_planta_casa.copy()

valores_eliminar = ["finca_rustica", "garaje", "garaje/local", "edificio"]

if "tipo_inmueble" in df_limpio.columns:
    df_limpio = df_limpio[~df_limpio["tipo_inmueble"].isin(valores_eliminar)].copy()

print(f"Registros eliminados = {len(df_subtipo_normalizado)-len(df_limpio)}")


Registros eliminados = 9


Si `tipo_inmueble` viene como 'ático', lo pasamos a `subtipo` y forzamos `tipo_inmueble = piso`.

In [12]:
df_atico = df_limpio.copy()

if "tipo_inmueble" in df_atico.columns and "subtipo" in df_atico.columns:
    mask_duplex_atico = df_atico["tipo_inmueble"].isin(["ático", "atico"])
    df_atico.loc[mask_duplex_atico, "subtipo"] = df_atico.loc[mask_duplex_atico, "tipo_inmueble"]
    df_atico.loc[mask_duplex_atico, "tipo_inmueble"] = "piso"

Unificación de categorías (chalet → casa)


In [13]:
df_chalet = df_atico.copy()

if "tipo_inmueble" in df_chalet.columns:
    df_chalet["tipo_inmueble"] = df_chalet["tipo_inmueble"].replace({"chalet": "casa"})

df_chalet.head()

,fuente,tipo_inmueble,subtipo,direccion_texto,barrio_zona,municipio,precio_eur,precio_anterior_eur,descuento_pct,garaje_incluido,habitaciones,superficie_m2,planta,exterior,ascensor,obra_nueva,finca
id_anuncio,,,,,,,,,,,,,,,,,
1,Inmobiliaria E-Castro,piso,NaN,"""Avenida Riomar, 1",Cotolino,Castro-Urdiales,498000.0,NaN,NaN,1.0,4.0,121.0,4.0,1.0,1.0,0.0,0
2,Inmobiliarias.com,casa,chalét independiente,"Barrio Brazomar, 46",Cotolino,Castro-Urdiales,575000.0,NaN,NaN,1.0,5.0,232.0,0.0,NaN,NaN,0.0,0
3,Inmobiliaria E-Castro,piso,NaN,"""Calle Leonardo Rucabado, 6",Centro,Castro-Urdiales,299000.0,NaN,NaN,NaN,2.0,84.0,3.0,1.0,1.0,0.0,0
4,Inmobiliaria E-Castro,piso,NaN,"""Calle de la Ronda, 32",Centro,Castro-Urdiales,340000.0,NaN,NaN,NaN,4.0,99.0,5.0,1.0,1.0,0.0,0
5,Inmobiliaria E-Castro,casa,chalét independiente,Barrio Sámano,Sámano,Castro-Urdiales,799000.0,NaN,NaN,1.0,5.0,511.0,0.0,NaN,NaN,0.0,0


Si es `piso` y no es ático/dúplex, asignamos subtipo por defecto `apartamento`.


In [14]:
df_apartamento = df_chalet.copy()

if "tipo_inmueble" in df_apartamento.columns and "subtipo" in df_apartamento.columns:
    mask_piso_sin_subtipo = (
        (df_apartamento["tipo_inmueble"] == "piso") &
        (~df_apartamento["subtipo"].isin(["dúplex", "duplex", "ático", "atico"]))
    )
    df_apartamento.loc[mask_piso_sin_subtipo & df_apartamento["subtipo"].isna(), "subtipo"] = "apartamento"


In [15]:
df_apartamento["subtipo"].unique() if "subtipo" in df_apartamento.columns else None

<StringArray>
[                'apartamento',        'chalét independiente',
                       'ático',                           nan,
              'chalét adosado',              'chalét pareado',
              'casa de piedra',                      'dúplex',
              'casa de pueblo',          'casa independiente',
                        'casa',                      'chalét',
                     'estudio',                       'masía',
                       'villa',              'casa montañesa',
                      'casona',                  'casa rural',
                'casa indiana',            'casa unifamiliar',
                'casa adosada',                     'palacio',
                       'torre', 'casa o chalet independiente',
              'chalet pareado', 'casa o chalet independent﻿e',
              'chalet adosado',               'finca rústica',
                      'chalet']
Length: 29, dtype: str

## 5) Mapeo fino de `subtipo`

Unificamos variantes de texto.


In [16]:
df_subtipo = df_apartamento.copy()

if "subtipo" in df_subtipo.columns:
    df_subtipo["subtipo"] = (
        df_subtipo["subtipo"]
            .astype(str)
            .str.lower()
            .str.strip()
            .replace({"nan": np.nan})
            .replace({
                "chalét independiente": "casa independiente",
                "chalet independiente": "casa independiente",
                "chalét pareado": "casa adosada",
                "chalet pareado": "casa adosada",
                "chalét adosado": "casa adosada",
                "chalet adosado": "casa adosada",
                "casa o chalet independiente": "casa independiente",
                "casa o chalet independiente\ufeff": "casa independiente",
                "chalet": "casa independiente",
                "casa unifamiliar": "casa independiente"
            })
    )


In [17]:
df_subtipo["subtipo"].unique() if "subtipo" in df_subtipo.columns else None

<StringArray>
[                'apartamento',          'casa independiente',
                       'ático',                           nan,
                'casa adosada',              'casa de piedra',
                      'dúplex',              'casa de pueblo',
                        'casa',                      'chalét',
                     'estudio',                       'masía',
                       'villa',              'casa montañesa',
                      'casona',                  'casa rural',
                'casa indiana',                     'palacio',
                       'torre', 'casa o chalet independent﻿e',
               'finca rústica']
Length: 21, dtype: str

Si falta `subtipo`:
- si `tipo_inmueble` es `casa` → `casa independiente`
- si `tipo_inmueble` es `piso` → `apartamento`


In [18]:
df_subtipo_relleno = df_subtipo.copy()

if "tipo_inmueble" in df_subtipo_relleno.columns and "subtipo" in df_subtipo_relleno.columns:
    df_subtipo_relleno.loc[(df_subtipo_relleno["subtipo"].isna()) & (df_subtipo_relleno["tipo_inmueble"] == "casa"), "subtipo"] = "casa independiente"
    df_subtipo_relleno.loc[(df_subtipo_relleno["subtipo"].isna()) & (df_subtipo_relleno["tipo_inmueble"] == "piso"), "subtipo"] = "apartamento"


## 6) Limpieza de `direccion_texto`

Elimina comillas sobrantes al inicio/fin.


In [19]:
df_direccion = df_subtipo_relleno.copy()

if "direccion_texto" in df_direccion.columns:
    df_direccion["direccion_texto"] = (
        df_direccion["direccion_texto"]
            .astype(str)
            .str.strip()
            .str.strip('"')
            .str.strip("'")
            .str.strip()
            .replace({"nan": np.nan})
    )


## 7) Tipado de variables numéricas + filtrado mínimo

Convertimos a numérico y descartamos filas sin `precio_eur`.


In [20]:
df_numerico = df_direccion.copy()

cols_numericas = [
    "precio_eur",
    "superficie_m2",
    "habitaciones",
    "planta"
    ]

for col in cols_numericas:
    if col in df_numerico.columns:
        df_numerico[col] = pd.to_numeric(df_numerico[col], errors="coerce")

for col in cols_numericas:
    if col in df_numerico.columns:
        print(col, df_numerico[col].isna().sum())


precio_eur 1
superficie_m2 96
habitaciones 49
planta 72


Descartamos la columna de descuentos y precio anterior

In [21]:
df_clean = df_numerico.copy()

if "precio_eur" in df_clean.columns:
    df_clean = df_clean[df_clean["precio_eur"].notna()].copy()

df_clean = df_clean.drop(columns=["precio_anterior_eur", "descuento_pct"], axis=1)

df_clean.head(20)

ValueError: Cannot specify both 'axis' and 'index'/'columns'

## 8) Valores faltantes en columnas clave

In [ ]:
print(df_clean.isna().sum())

registros_sin_superficie = df_clean["superficie_m2"].isna().sum()

print(f"\nHay {registros_sin_superficie}/{len(df_clean)} sin superficie ({round(registros_sin_superficie*100/len(df_clean),2)} %)\n")

In [ ]:
df_clean[df_clean["superficie_m2"].isna()].describe()

In [ ]:
print(df_clean["obra_nueva"].value_counts())

Los 95 registros sin superficie son obra nueva. En total, 184 registros del dataset son obra nueva.
Se separan en 2 datasets distintos.

In [ ]:
df_segunda_mano = (
    df_clean[df_clean["obra_nueva"] == 0]
    .drop(columns=["obra_nueva"])
    .copy()
)

df_obra_nueva = (
    df_clean[df_clean["obra_nueva"] == 1]
    .drop(columns=["obra_nueva"])
    .copy()
)

## 8) Checks rápidos

In [ ]:
print(f"Dataset de viviendas de segunda mano: {df_segunda_mano.shape}")
print(f"Dataset de viviendas de obra nueva: {df_obra_nueva.shape}")

In [ ]:
df_segunda_mano.isna().sum()

In [ ]:
df_obra_nueva.isna().sum()

## 9) Exportación de los datasets limpios para el siguiente paso

In [ ]:
output_dir = Path("../data/1st_cleaning")
output_dir.mkdir(parents=True, exist_ok=True)

df_segunda_mano.to_csv(output_dir / "segunda_mano_clean.csv", index=False)
df_obra_nueva.to_csv(output_dir / "obra_nueva_clean.csv", index=False)